# DGN Data Generation Tutorial

This notebook demonstrates how to generate synthetic datasets using data-generating networks (DGN), an ensemble of RNN modules jointly trained to perform specific cognitive tasks, where each module represents a distinct brain region. We will explore how to use `generate_data.py` and provide an overview of how experiment config files are organized.

You will learn:
- how to run each example experiment (`memory_network`, `pass_decision`, `multi_task`)
- where outputs are saved (`data.h5`, checkpoints, logs)
- how `main.yaml`, `model`, `datamodule`, and `callbacks` configs work together
- how to create your own experiment and generate your own data

## 1) Config layout

Each experiment lives under `configs/<experiment_name>/`.

Expected files:
- `main.yaml` (top-level experiment config)
    - picks which `model`, `datamodule`, `callbacks` sub-configs to compose
    - defines trainer/logger defaults
- `model/model.yaml`
    - sets model class and model hyperparameters
- `datamodule/datamodule.yaml` (data generator config)
    - controls synthetic data distribution, dimensions, and train/val splits
- `callbacks/callbacks.yaml`
    - adds logging/plots and H5 export behavior (`SaveAsH5`)

In [ ]:
from pathlib import Path
repo_root = Path.cwd().parent.parent

config_root = repo_root / "configs"
experiments = sorted([p.name for p in config_root.iterdir() if p.is_dir()])
print("Default experiments:", experiments)

for exp in experiments:
    exp_dir = config_root / exp
    print(f"\n[{exp}]")
    for rel in [
        "main.yaml",
        "model/model.yaml",
        "datamodule/datamodule.yaml",
        "callbacks/callbacks.yaml",
    ]:
        path = exp_dir / rel
        print(" -", rel, "OK" if path.exists() else "MISSING")

Default experiments: ['memory_network', 'multi_task', 'pass_decision']

[memory_network]
 - main.yaml OK
 - model/model.yaml OK
 - datamodule/datamodule.yaml OK
 - callbacks/callbacks.yaml OK

[multi_task]
 - main.yaml OK
 - model/model.yaml OK
 - datamodule/datamodule.yaml OK
 - callbacks/callbacks.yaml OK

[pass_decision]
 - main.yaml OK
 - model/model.yaml OK
 - datamodule/datamodule.yaml OK
 - callbacks/callbacks.yaml OK


## 2) Run generation from CLI

The universal command is:

```bash
python3 generate_data.py <experiment_name>
```

Examples:
- `python3 generate_data.py memory_network`
- `python3 generate_data.py pass_decision`
- `python3 generate_data.py multi_task`

Available Flags:
- `--run_name my_custom_run`
- `--seed 0`
- `--config /absolute/path/to/main.yaml`
- `--overrides trainer.max_epochs=5 model.hidden_size=32`

In [2]:
import subprocess
import os

# TODO: fix pkg_resources warning instead of ignoring
env = os.environ.copy()
env["PYTHONWARNINGS"] = "ignore"

cmd = [
    "python3", "generate_data.py", "memory_network",
    "--run_name", "tutorial_memory_network",
    "--overrides", "trainer.max_epochs=1", "trainer.min_epochs=1"
]

print("Running:", " ".join(cmd))

subprocess.run(cmd, cwd=repo_root, check=True, env=env)

Running: python3 -m dgn.generate_data memory_network --run_name tutorial_memory_network --overrides trainer.max_epochs=1 trainer.min_epochs=1


Global seed set to 0
ModelCheckpoint(save_last=True, save_top_k=-1, monitor=None) will duplicate the last checkpoint saved.
GPU available: False, used: False
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs

  | Name    | Type       | Params
---------------------------------------
0 | areas   | ModuleDict | 47.9 K
1 | mseloss | MSELoss    | 0     
---------------------------------------
47.9 K    Trainable params
0         Non-trainable params
47.9 K    Total params
0.192     Total estimated model params size (MB)


Epoch 0:  94%|█████████▍| 16/17 [00:35<00:02,  2.24s/it, loss=0.0283, v_num=]
Validation: 0it [00:00, ?it/s]
Epoch 0: 100%|██████████| 17/17 [01:02<00:00,  3.68s/it, loss=0.0283, v_num=]Saving data from epoch  0 ...
OVERRIDE data.h5
Epoch 0: 100%|██████████| 17/17 [01:02<00:00,  3.68s/it, loss=0.0283, v_num=]


CompletedProcess(args=['python3', '-m', 'dgn.generate_data', 'memory_network', '--run_name', 'tutorial_memory_network', '--overrides', 'trainer.max_epochs=1', 'trainer.min_epochs=1'], returncode=0)

## 3) Where outputs are saved

By default, outputs go to:
- `runs/<experiment>_id<timestamp>/`

If `--run_name` is provided:
- `runs/<run_name>/`

Key files produced:
- `data.h5` (exported by `SaveAsH5` callback)
- `lightning_checkpoints/*.ckpt`
- TensorBoard logs
- optional plots (e.g., under `graphs/`)

In [3]:
import h5py

run_dir = repo_root / "runs" / "tutorial_memory_network"
h5_path = run_dir / "data.h5"

print("H5 path:", h5_path)
print("Exists:", h5_path.exists())

if h5_path.exists():
    with h5py.File(h5_path, "r") as f:
        print("Top-level groups:", list(f.keys()))
        if "0" in f:
            ds_names = list(f["0"].keys())
            print("Session 0 datasets:", ds_names)

H5 path: /Users/william_ong/Desktop/Projects/mrlfads2/dgn/runs/tutorial_memory_network/data.h5
Exists: True
Top-level groups: ['0']
Session 0 datasets: ['area-A0', 'area-A1', 'area-A2', 'inputs-A0', 'inputs-A1', 'inputs-A2', 'message-latents', 'message-mesgs', 'truth-inp']


In [4]:
# # Optional: quickly print the active YAMLs for these two experiments
# from pathlib import Path

# cfg = repo_root / "configs"
# files = [
#     cfg / "memory_network" / "model" / "model.yaml",
#     cfg / "memory_network" / "datamodule" / "datamodule.yaml",
#     cfg / "pass_decision" / "model" / "model.yaml",
#     cfg / "pass_decision" / "datamodule" / "datamodule.yaml",
# ]

# for p in files:
#     print(f"\n===== {p.relative_to(repo_root)} =====")
#     print(p.read_text())